# 05 M3: deep hedging under Heston, stock only (c = 0)
Incomplete market: the stock alone cannot hedge volatility risk. Features add $V_k/b$. Compared with BS delta (fixed $\sigma$ and current $\sqrt{V_k}$) under $\mathrm{CVaR}_{0.5}$ and $\mathrm{CVaR}_{0.99}$.

In [3]:
# notebooks live in notebooks/, the project modules in the repo root
import sys
if '..' not in sys.path:
    sys.path.insert(0, '..')

%load_ext autoreload
%autoreload 2

# standard libraries
import math

# third-party
import numpy as np
import pandas as pd
import torch
import seaborn as sns
from matplotlib import pyplot as plt

# project modules
import benchmarks as bm
from model import Policy
from market.simulators import simulate_heston, simulate_gbm, HestonParams
from market.pricing import bs_price, bs_delta, bs_gamma, heston_price
from features import nn_policy, ntb_policy
from train import train
from objectives import EntropicLoss, CVaRLoss


*Constants and the feature-normalization dict shared by training and evaluation.*

In [4]:
# shared constants of the M3 experiments
K, T = 100.0, 30 / 365
params = HestonParams()                      # paper parameters

# feature normalization: one dict, used for training AND evaluation
feat_kw = dict(s_scale=(params.b * T) ** 0.5, use_var=True, v_scale=params.b)

*Helpers: fixed Heston test set (seed 0), CVaR price of a policy, comparison table; BS benchmark policies.*

In [7]:
with torch.no_grad():
    torch.manual_seed(0)
    test = simulate_heston(100_000, params=params)      # fixed Heston test set


def eval_cvar(pol, alpha, n_instr=1, chunk=10_000):
    """Price = CVaR_alpha of the loss on the Heston test set, plus the P&L summary.

    n_instr=1: hedge with the stock only;  n_instr=2: stock + variance swap.
    Paths are processed in chunks to limit memory (heston_greeks builds a (paths x 512) matrix).
    """
    S_all = test["S"][:, :, 0] if n_instr == 1 else test["S"][:, :, :n_instr]
    pnls = []
    with torch.no_grad():
        for i in range(0, S_all.shape[0], chunk):
            pnls.append(bm.hedge_pnl(S_all[i:i + chunk], pol, K, T,
                                     V=test["V"][i:i + chunk]))      # p0 = 0
    pnl = torch.cat(pnls)
    p0 = bm.summarize(pnl, alpha)["CVaR"]
    return {"p0": p0, **bm.summarize(pnl + p0, alpha)}


def cvar_table(pols, alpha):
    """One row per policy: price and P&L summary at CVaR level alpha."""
    return pd.DataFrame({name: eval_cvar(pol, alpha) for name, pol in pols.items()}).T


bs_pol = bm.bs_delta_policy(K, 0.2)            # BS delta, fixed sigma = sqrt(b)
bs_SV_pol = bm.bs_delta_stovol_policy(K)       # BS delta with the current sqrt(V_k)

*Train the M3 network with $\mathrm{CVaR}_{0.5}$ (features: log-moneyness, time, $V_k/b$).*

In [8]:
model_m3 = Policy(d_in = 3, m_out = 1)
out_m3 = train(model_m3, K, T, loss_fn=CVaRLoss(0.5), simulator=simulate_heston,
               sim_kwargs={"params": params}, n_steps=3000, **feat_kw)

step: 0, loss: 3.8146047592163086
step: 100, loss: 2.648847818374634
step: 200, loss: 2.5093960762023926
step: 300, loss: 2.496878147125244
step: 400, loss: 2.5559182167053223
step: 500, loss: 2.4453678131103516
step: 600, loss: 2.5850024223327637
step: 700, loss: 2.5728752613067627
step: 800, loss: 2.4988327026367188
step: 900, loss: 2.5033750534057617
step: 1000, loss: 2.515143871307373
step: 1100, loss: 2.501758098602295
step: 1200, loss: 2.564718008041382
step: 1300, loss: 2.5341930389404297
step: 1400, loss: 2.4892826080322266
step: 1500, loss: 2.488433361053467
step: 1600, loss: 2.4943981170654297
step: 1700, loss: 2.519071578979492
step: 1800, loss: 2.48807954788208
step: 1900, loss: 2.4795007705688477
step: 2000, loss: 2.533787727355957
step: 2100, loss: 2.5415382385253906
step: 2200, loss: 2.552278518676758
step: 2300, loss: 2.5369927883148193
step: 2400, loss: 2.512850761413574
step: 2500, loss: 2.5248003005981445
step: 2600, loss: 2.5472636222839355
step: 2700, loss: 2.54172

*$\mathrm{CVaR}_{0.5}$ prices on the test set: NN vs BS delta with $\sqrt{V_k}$ vs fixed $\sigma$.*

In [9]:
cvar_table({"NN 50":    nn_policy(model_m3, K, T, **feat_kw),
            "BS_SV 50": bs_SV_pol,
            "BS 50":    bs_pol}, 0.5)

,p0,mean,std,VaR,CVaR
NN 50,2.510407,0.814174,1.222172,-1.097875,-2.297974e-07
BS_SV 50,2.670661,0.976190,1.349290,-1.455262,-2.822876e-07
BS 50,2.658308,0.961645,1.342169,-1.377534,-2.307129e-07


*Train a second network with $\mathrm{CVaR}_{0.99}$ (tail risk; only ~50 tail paths per batch, so training loss is noisy -- trust the test set).*

In [10]:
model_m3_99 = Policy(d_in = 3, m_out = 1)
out_m3_99 = train(model_m3_99, K, T, loss_fn=CVaRLoss(0.99), simulator=simulate_heston,
                  sim_kwargs={"params": params}, n_steps=3000, **feat_kw)

step: 0, loss: 167.44207763671875
step: 100, loss: 85.05402374267578
step: 200, loss: 49.57563400268555
step: 300, loss: 29.687942504882812
step: 400, loss: 21.95783233642578
step: 500, loss: 15.73632526397705
step: 600, loss: 12.149993896484375
step: 700, loss: 10.912595748901367
step: 800, loss: 9.799797058105469
step: 900, loss: 8.827329635620117
step: 1000, loss: 7.77175235748291
step: 1100, loss: 6.8578290939331055
step: 1200, loss: 6.9853596687316895
step: 1300, loss: 7.190692901611328
step: 1400, loss: 6.599408149719238
step: 1500, loss: 6.910643577575684
step: 1600, loss: 6.655040740966797
step: 1700, loss: 5.862268447875977
step: 1800, loss: 6.304220199584961
step: 1900, loss: 6.44282341003418
step: 2000, loss: 5.98937463760376
step: 2100, loss: 6.386159420013428
step: 2200, loss: 5.940295219421387
step: 2300, loss: 5.931430339813232
step: 2400, loss: 5.788710594177246
step: 2500, loss: 5.658160209655762
step: 2600, loss: 5.664754867553711
step: 2700, loss: 6.25907564163208
st

*$\mathrm{CVaR}_{0.99}$ prices: in the tail the current-vol information matters much more (fixed-$\sigma$ BS is far worse).*

In [11]:
cvar_table({"NN 99":    nn_policy(model_m3_99, K, T, **feat_kw),
            "BS_SV 99": bs_SV_pol,
            "BS 99":    bs_pol}, 0.99)

,p0,mean,std,VaR,CVaR
NN 99,5.908487,4.212964,1.296902,-0.761223,4.239082e-07
BS_SV 99,6.743778,5.049307,1.349290,-0.762300,-1.811981e-07
BS 99,7.914933,6.218271,1.342169,-1.528810,1.177788e-07
